<a href="https://colab.research.google.com/github/nativemind11/NativeMind-Tasks/blob/main/Copy_of_NLP_project_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

** Importing Libraries + Downloading NLTK Stopwords**

In [2]:
import string
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from transformers import pipeline
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


** Dataset - 8 Sentences (Arabic + English) with Labels**

In [3]:
texts = [ "I love this movie, it was amazing",
         "This film was terrible and boring",
          "Great acting and wonderful story",
          "Waste of time, very bad movie",
          "هذا الفيلم رائع جدا وممتع" ,
          "هذا الفيلم سيء للغاية ومملل" ,
          "األداء التمثيلي كان ممتاز جدا" ,"القصة كانت ضعيفة وغير مقنعة"  ]
labels = [1, 0, 1, 0, 1, 0, 1, 0]

** Preprocessing Function (Tokenization + Normalization + Arabic & English Stopword Removal via NLTK)**

In [4]:
def preprocess(text):
  text =text.lower()
  for punct in string.punctuation:
      text =text.replace(punct, "")
  tokens =text.split()
  stop_words = stopwords.words('english') + stopwords.words('arabic')
  tokens = [t for t in tokens if t not in stop_words]
  return " ".join(tokens)

**Feature Extraction (Bag of Words) + Training & Testing a Naive Bayes **

In [5]:
cleaned_texts = [preprocess(t) for t in texts]
print("Cleaned:", cleaned_texts)

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(cleaned_texts) #BOW

model = MultinomialNB() #Naive Bayes
model.fit(X ,labels)

test = ["I really  enjoyed this film"] #test  (enjoyed first time ) # دقة محدودة متوقعة لصغر حجم بيانات التدريب (8 جمل) - مقارن بموديل Transformer الأدق في الخلية اللي بعدها
test_X = vectorizer.transform([preprocess(test[0])])
print("Naive Bayes prediction:", model.predict(test_X))

Cleaned: ['love movie amazing', 'film terrible boring', 'great acting wonderful story', 'waste time bad movie', 'الفيلم رائع جدا وممتع', 'الفيلم سيء للغاية ومملل', 'األداء التمثيلي ممتاز جدا', 'القصة كانت ضعيفة وغير مقنعة']
Naive Bayes prediction: [0]


** Pretrained Transformer Model (HuggingFace) + Accuracy Comparison Between Arabic and English**

In [6]:
classifier = pipeline("sentiment-analysis")
result = classifier(texts)
for text, res in zip(texts, result):
    print(f"{text} → {res['label']} ({res['score']:.2f})")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

I love this movie, it was amazing → POSITIVE (1.00)
This film was terrible and boring → NEGATIVE (1.00)
Great acting and wonderful story → POSITIVE (1.00)
Waste of time, very bad movie → NEGATIVE (1.00)
هذا الفيلم رائع جدا وممتع → NEGATIVE (0.67)
هذا الفيلم سيء للغاية ومملل → NEGATIVE (0.66)
األداء التمثيلي كان ممتاز جدا → NEGATIVE (0.63)
القصة كانت ضعيفة وغير مقنعة → NEGATIVE (0.72)


**Custom RNN Model Built and Trained from Scratch (PyTorch)**

In [7]:
vocab = {"good": 1, "bad": 2, "great": 3, "terrible": 4, "amazing": 5, "boring": 6, "movie": 7, "": 0}

def encode(text, vocab, max_len=5):
    words = text.lower().split()
    ids = [vocab.get(w, 0) for w in words][:max_len]
    ids += [0] * (max_len - len(ids))
    return ids

simple_texts = ["good movie amazing", "bad movie terrible", "great movie", "boring movie terrible"]
simple_labels = [1, 0, 1, 0]

X_rnn = torch.tensor([encode(t, vocab) for t in simple_texts])
y_rnn = torch.tensor(simple_labels)

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=8, hidden_dim=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        return self.fc(hidden.squeeze(0))

model_rnn = SimpleRNN(vocab_size=len(vocab))
optimizer = torch.optim.Adam(model_rnn.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(50):
    optimizer.zero_grad()
    output = model_rnn(X_rnn)
    loss = loss_fn(output, y_rnn)
    loss.backward()
    optimizer.step()

print("RNN تدريب خلص، آخر قيمة خطأ:", loss.item())

RNN تدريب خلص، آخر قيمة خطأ: 0.007018167991191149
